# Plastic Waste Processing: Mass Balance and Project Economics

This walkthrough demonstrates a simplified process mass balance and discounted cash-flow model. It uses illustrative assumptions, not verified Bangladesh market quotations or a commercial investment case.

In [ ]:
from pathlib import Path
import sys,json
import pandas as pd
ROOT=Path.cwd()
if not (ROOT/"src").exists(): ROOT=ROOT.parent
sys.path.insert(0,str(ROOT/"src"))
from plastic_waste_economics.model import (
    project_cash_flows, npv, irr, simple_payback_years
)
cfg=json.loads((ROOT/"config/base_case.json").read_text())
cfg

## 1. Material balance and revenues

In [ ]:
model=project_cash_flows(**cfg)
display(pd.Series(model["mass_balance"],name="tonnes").to_frame())
print("Annual revenue (steady-state):",round(model["annual_revenue_at_steady_state"],2))
print("Annual variable OPEX (steady-state):",round(model["annual_variable_opex_at_steady_state"],2))
print("Annual fixed OPEX:",round(model["annual_fixed_opex"],2))
print("Annual EBITDA (steady-state):",round(model["annual_ebitda_at_steady_state"],2))


## 2. Discounted cash-flow metrics

In [ ]:
flows=model["cash_flows"]
summary={
    "NPV":npv(flows,cfg["discount_rate"]),
    "IRR":irr(flows),
    "Simple payback years":simple_payback_years(flows)
}
display(pd.Series(summary,name="value").to_frame())

## 3. Sensitivity questions

Stress-test recovery yield, feedstock price, product sale price, gate fee, energy and transport costs, capacity utilisation and capex. A positive base-case NPV does not guarantee feasibility: verify feedstock contracts, product offtake, residues, regulatory approvals, working capital and environmental controls.

All default inputs are illustrative only.

## 4. Understand the mass balance

The revised model separates the recovered stream from the saleable product. Some recovered material may fail the product specification and is counted as a residual. A closed mass balance is an accounting check, not proof that the selected process technology can achieve the assumed yield.


In [ ]:
mass = pd.Series(model["mass_balance"], name="tonnes")
display(mass.to_frame())
assert abs(model["mass_balance"]["mass_balance_residual_tonnes"]) < 1e-9
print("Incoming mass equals saleable product plus grouped residual mass.")


## 5. Review annual cash flow and break-even gate fee

The model now allows a ramp-up period, replacement costs and salvage value. The tax calculation is deliberately simplified: straight-line depreciation, tax on positive EBIT and no carry-forward of tax losses. It excludes financing and working capital.


In [ ]:
cash_table = pd.DataFrame(model["annual_cash_flow_table"])
display(cash_table.round(2))

gate_fee = break_even_gate_fee_per_tonne(cfg)
print("Break-even gate fee (per incoming tonne):", round(gate_fee, 2))
print("This is a model-derived threshold, not evidence that a supplier or municipality will pay it.")


## 6. One-way and two-way sensitivity

One-way sensitivity changes a single input while leaving the rest of the base case unchanged. Two-way sensitivity shows how two assumptions interact. These are deterministic stress tests; they are not probability distributions or forecasts.


In [ ]:
from plastic_waste_economics.sensitivity import (
    one_way_sensitivity,
    two_way_npv_sensitivity,
)

one_way = one_way_sensitivity(
    cfg, "recovery_yield", [0.50, 0.60, 0.70, 0.80, 0.90]
)
display(one_way.round(2))

two_way = two_way_npv_sensitivity(
    cfg,
    "recovery_yield", [0.50, 0.60, 0.70, 0.80],
    "product_price_per_tonne", [300, 400, 500, 600, 700],
)
pivot = two_way.pivot(
    index="recovery_yield",
    columns="product_price_per_tonne",
    values="npv",
)
display(pivot.round(0))

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(one_way["value"], one_way["npv"], marker="o")
ax.axhline(0, linestyle="--")
ax.set(
    title="NPV sensitivity to recovery yield",
    xlabel="Recovery yield (illustrative fraction)",
    ylabel="NPV (illustrative currency units)",
)
ax.grid(alpha=0.2)
plt.tight_layout()
plt.show()


## 7. Decision checklist

Before interpreting a positive NPV, verify feedstock contracts and polymer composition, pilot-scale mass balance, product quality and off-take terms, vendor quotations, uptime, utilities, residue management, environmental permits, worker health and safety, working capital and tax treatment. Do not select a conversion technology or claim environmental benefit based on a financial spreadsheet alone.
